# VeriPromiseESG 2026 — 最終版 Pipeline（Kaggle）

內建所有經實測有效的決策：
- 合併官方 train + val = 2000 筆訓練資料（最大功臣）
- 截斷式類別權重（壓掉 Misleading 1 筆的爆量權重）
- 邏輯約束解碼（四欄位組合永遠合法）
- macbert-base 多 seed × 5 折集成（EPOCHS=8、FGM、fp16）
- Macro-F1 門檻優化（純後處理，針對稀有類別）

> 右側 Settings → Accelerator 選 **GPU T4 x2**；Add Input 掛上含三個 json 的 dataset。
> 路徑會自動偵測。完整跑完約 60–75 分鐘，建議用 **Save & Run All (Commit)** 讓它跑完自動關。


## Step 0｜安裝套件


In [ ]:
# 掛載 Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install -q transformers==4.44.2 2>/dev/null
print("ok")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 62.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 37.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 56.6 MB/s eta 0:00:00
ok


## Step 1｜匯入與超參數


In [ ]:
import os, json, glob, random, pickle, warnings
import numpy as np, pandas as pd
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, classification_report
from sklearn.utils.class_weight import compute_class_weight
warnings.filterwarnings("ignore")

# ===== 超參數 =====
MAX_LEN          = 384
BATCH_SIZE       = 16
EPOCHS           = 8
LR               = 2e-5
CLASS_WEIGHT_MAX = 5.0
USE_FGM          = True
USE_AMP          = True
SEED             = 42
MODELS           = ["hfl/chinese-macbert-base"]   # 可加 backbone，但實測 base 最穩
SEEDS            = [42, 1234, 2025]                # 多 seed 集成
N_SPLITS         = 5
OUT_DIR          = "/content/drive/MyDrive/AICUP"

EVAL_FIELDS = {
    "promise_status": ["Yes","No"],
    "verification_timeline": ["already","within_2_years","between_2_and_5_years","more_than_5_years","N/A"],
    "evidence_status": ["Yes","No","N/A"],
    "evidence_quality": ["Clear","Not Clear","Misleading","N/A"],
}
FIELD_WEIGHTS = {"promise_status":0.20,"verification_timeline":0.15,"evidence_status":0.30,"evidence_quality":0.35}
label2id   = {f:{l:i for i,l in enumerate(ls)} for f,ls in EVAL_FIELDS.items()}
num_labels = {f:len(ls) for f,ls in EVAL_FIELDS.items()}

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("裝置:", device, "| EPOCHS:", EPOCHS, "| SEEDS:", SEEDS)

裝置: cuda | EPOCHS: 8 | SEEDS: [42, 1234, 2025]


## Step 2｜載入資料（合併 train + val = 2000 筆）
路徑自動偵測，不管你的 dataset 取什麼名字都找得到。


In [ ]:
def find_file(name):
    hits = glob.glob(f"/content/drive/MyDrive/AICUP/{name}", recursive=True)
    if not hits: raise FileNotFoundError(f"找不到 {name}，請確認 dataset 已掛上 (Add Input)")
    return hits[0]

TRAIN_PATH = find_file("vpesg4k_train_1000.json")
VAL_PATH   = find_file("vpesg4k_val_1000.json")
TEST_PATH  = find_file("vpesg4k_test_2000.json")

with open(TRAIN_PATH, "r", encoding="utf-8") as f: train_part = json.load(f)
with open(VAL_PATH,   "r", encoding="utf-8") as f: val_part   = json.load(f)
train_data = train_part + val_part                       # 1000 + 1000 = 2000
with open(TEST_PATH,  "r", encoding="utf-8") as f: test_data  = json.load(f)

# 安全檢查：訓練 id 不可與測試 id 重疊
tr_ids=set(int(x["id"]) for x in train_data); te_ids=set(int(x["id"]) for x in test_data)
assert len(tr_ids & te_ids)==0, "訓練集與測試集 id 重疊！"
assert len(test_data)==2000 and int(test_data[0]["id"])==12001, "測試集應為 2000 筆、id 由 12001 起"

print(f"訓練 {len(train_data)} 筆 (train {len(train_part)} + val {len(val_part)}) | 測試 {len(test_data)} 筆")
from collections import Counter
for f in EVAL_FIELDS:
    c = Counter(s[f] for s in train_data)
    print(f"[{f}]", {k: c.get(k,0) for k in EVAL_FIELDS[f]})

訓練 2000 筆 (train 1000 + val 1000) | 測試 2000 筆
[promise_status] {'Yes': 1627, 'No': 373}
[verification_timeline] {'already': 718, 'within_2_years': 34, 'between_2_and_5_years': 498, 'more_than_5_years': 377, 'N/A': 373}
[evidence_status] {'Yes': 1345, 'No': 282, 'N/A': 373}
[evidence_quality] {'Clear': 1118, 'Not Clear': 225, 'Misleading': 2, 'N/A': 655}


## Step 3｜Dataset / 模型 / FGM / 工具函式


In [ ]:
class ESGDataset(Dataset):
    def __init__(self, data, tokenizer, has_labels=True):
        self.data=data; self.tok=tokenizer; self.has_labels=has_labels
    def __len__(self): return len(self.data)
    def __getitem__(self, idx):
        s=self.data[idx]
        enc=self.tok(s["data"], truncation=True, max_length=MAX_LEN, padding="max_length", return_tensors="pt")
        item={"input_ids":enc["input_ids"].squeeze(0), "attention_mask":enc["attention_mask"].squeeze(0)}
        if self.has_labels:
            item["labels"]={f:torch.tensor(label2id[f][s[f]],dtype=torch.long) for f in EVAL_FIELDS}
        return item

class MultiTaskModel(nn.Module):
    def __init__(self, model_name, num_labels_dict, dropout=0.2):
        super().__init__()
        self.backbone=AutoModel.from_pretrained(model_name)
        h=self.backbone.config.hidden_size
        self.dropout=nn.Dropout(dropout)
        self.heads=nn.ModuleDict({f:nn.Linear(h,n) for f,n in num_labels_dict.items()})
    def forward(self, input_ids, attention_mask):
        out=self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        last=out.last_hidden_state
        mask=attention_mask.unsqueeze(-1).float()
        pooled=(last*mask).sum(1)/mask.sum(1).clamp(min=1e-9)   # masked mean pooling
        return {f:head(self.dropout(pooled)) for f,head in self.heads.items()}

class FGM:
    def __init__(self, model, eps=1.0): self.model=model; self.eps=eps; self.backup={}
    def attack(self, name="word_embeddings"):
        for n,p in self.model.named_parameters():
            if p.requires_grad and name in n and p.grad is not None:
                self.backup[n]=p.data.clone()
                norm=torch.norm(p.grad)
                if norm!=0 and not torch.isnan(norm): p.data.add_(self.eps*p.grad/norm)
    def restore(self, name="word_embeddings"):
        for n,p in self.model.named_parameters():
            if p.requires_grad and name in n and n in self.backup: p.data=self.backup[n]
        self.backup={}

def compute_class_weights(data):
    cw={}
    for f,labels in EVAL_FIELDS.items():
        y=[label2id[f][s[f]] for s in data]
        present=np.unique(y); w=np.ones(len(labels),dtype=np.float32)
        for c,val in zip(present, compute_class_weight("balanced",classes=present,y=y)): w[c]=val
        w=np.clip(w, None, CLASS_WEIGHT_MAX)
        cw[f]=torch.tensor(w,dtype=torch.float32).to(device)
    return cw

def train_one_epoch(model, loader, optim, sched, crits, scaler, fgm):
    model.train(); total=0
    for batch in loader:
        ids=batch["input_ids"].to(device); am=batch["attention_mask"].to(device)
        labels={f:batch["labels"][f].to(device) for f in EVAL_FIELDS}
        optim.zero_grad()
        with torch.cuda.amp.autocast(enabled=USE_AMP):
            logits=model(ids,am); loss=sum(crits[f](logits[f],labels[f]) for f in EVAL_FIELDS)
        scaler.scale(loss).backward()
        if fgm is not None:
            fgm.attack()
            with torch.cuda.amp.autocast(enabled=USE_AMP):
                logits=model(ids,am); loss2=sum(crits[f](logits[f],labels[f]) for f in EVAL_FIELDS)
            scaler.scale(loss2).backward(); fgm.restore()
        scaler.unscale_(optim); torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
        scaler.step(optim); scaler.update(); sched.step(); total+=loss.item()
    return total/len(loader)

@torch.no_grad()
def predict_probs(model, loader):
    model.eval(); probs={f:[] for f in EVAL_FIELDS}
    for batch in loader:
        ids=batch["input_ids"].to(device); am=batch["attention_mask"].to(device)
        with torch.cuda.amp.autocast(enabled=USE_AMP): logits=model(ids,am)
        for f in EVAL_FIELDS: probs[f].append(torch.softmax(logits[f].float(),1).cpu().numpy())
    return {f:np.concatenate(probs[f],0) for f in EVAL_FIELDS}

def decode_with_logic(probs):
    N=len(probs["promise_status"]); out=[]
    yes,no=label2id["promise_status"]["Yes"],label2id["promise_status"]["No"]
    tl=["already","within_2_years","between_2_and_5_years","more_than_5_years"]; tl_ids=[label2id["verification_timeline"][l] for l in tl]
    es=["Yes","No"]; es_ids=[label2id["evidence_status"][l] for l in es]
    eq=["Clear","Not Clear","Misleading"]; eq_ids=[label2id["evidence_quality"][l] for l in eq]
    for i in range(N):
        ps="Yes" if probs["promise_status"][i,yes]>=probs["promise_status"][i,no] else "No"
        r={"promise_status":ps}
        if ps=="No":
            r.update(verification_timeline="N/A",evidence_status="N/A",evidence_quality="N/A")
        else:
            r["verification_timeline"]=tl[int(np.argmax(probs["verification_timeline"][i,tl_ids]))]
            e=es[int(np.argmax(probs["evidence_status"][i,es_ids]))]; r["evidence_status"]=e
            r["evidence_quality"]=eq[int(np.argmax(probs["evidence_quality"][i,eq_ids]))] if e=="Yes" else "N/A"
        out.append(r)
    return out

def weighted_macro_f1(gt, pred):
    score=0.0; detail={}
    for f,labels in EVAL_FIELDS.items():
        mf1=f1_score([g[f] for g in gt],[p[f] for p in pred],labels=labels,average="macro",zero_division=0)
        detail[f]=mf1; score+=mf1*FIELD_WEIGHTS[f]
    return score, detail

print("元件就緒")

元件就緒


## Step 4｜多 seed × 5 折集成訓練
每折以本地驗證加權 F1 選最佳 epoch；測試集機率跨所有模型平均，OOF 跨 seed 平均。
跑完自動存 oof/test 機率到 /kaggle/working，方便之後重跑門檻優化不必再訓練。


In [ ]:
bb = MODELS[0]
strat = [s["verification_timeline"] for s in train_data]
test_prob_sum={f:np.zeros((len(test_data),num_labels[f])) for f in EVAL_FIELDS}
oof_sum={f:np.zeros((len(train_data),num_labels[f])) for f in EVAL_FIELDS}
test_count=0

for si,sd in enumerate(SEEDS):
    print(f"\n========== SEED {sd}  ({si+1}/{len(SEEDS)}) ==========")
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=sd)
    folds = list(skf.split(range(len(train_data)), strat))
    tokenizer = AutoTokenizer.from_pretrained(bb)
    test_loader = DataLoader(ESGDataset(test_data, tokenizer, has_labels=False), batch_size=BATCH_SIZE, shuffle=False)
    oof_this={f:np.zeros((len(train_data),num_labels[f])) for f in EVAL_FIELDS}

    for fold,(tr_idx,va_idx) in enumerate(folds):
        print(f"\n=== SEED {sd} | Fold {fold+1}/{N_SPLITS} ===")
        tr=[train_data[i] for i in tr_idx]; va=[train_data[i] for i in va_idx]
        tr_loader=DataLoader(ESGDataset(tr,tokenizer),batch_size=BATCH_SIZE,shuffle=True)
        va_loader=DataLoader(ESGDataset(va,tokenizer,has_labels=False),batch_size=BATCH_SIZE,shuffle=False)
        set_seed(sd*100+fold)
        model=MultiTaskModel(bb,num_labels).to(device)
        crits={f:nn.CrossEntropyLoss(weight=compute_class_weights(tr)[f]) for f in EVAL_FIELDS}
        optim=torch.optim.AdamW(model.parameters(),lr=LR,weight_decay=0.01)
        total_steps=len(tr_loader)*EPOCHS
        sched=get_linear_schedule_with_warmup(optim,int(0.1*total_steps),total_steps)
        scaler=torch.cuda.amp.GradScaler(enabled=USE_AMP)
        fgm=FGM(model) if USE_FGM else None
        best=-1; best_probs=None; best_test=None
        for ep in range(EPOCHS):
            loss=train_one_epoch(model,tr_loader,optim,sched,crits,scaler,fgm)
            vp=predict_probs(model,va_loader)
            sc,_=weighted_macro_f1(va, decode_with_logic(vp))
            print(f"  epoch {ep+1}: loss={loss:.3f}  val加權F1={sc:.4f}")
            if sc>best: best=sc; best_probs=vp; best_test=predict_probs(model,test_loader)
        for f in EVAL_FIELDS:
            oof_this[f][va_idx]=best_probs[f]; test_prob_sum[f]+=best_test[f]
        test_count+=1
        del model; torch.cuda.empty_cache()

    sc_sd,_=weighted_macro_f1(train_data, decode_with_logic(oof_this))
    print(f"\n>>> SEED {sd} OOF CV = {sc_sd:.4f}")
    for f in EVAL_FIELDS: oof_sum[f]+=oof_this[f]

test_prob={f:test_prob_sum[f]/test_count for f in EVAL_FIELDS}
oof_prob ={f:oof_sum[f]/len(SEEDS) for f in EVAL_FIELDS}

with open(f"{OUT_DIR}/probs_ensemble.pkl","wb") as f:
    pickle.dump({"oof":oof_prob,"test":test_prob}, f)
print("\n集成完成，機率已存檔 probs_ensemble.pkl")


========== SEED 42  (1/3) ==========


tokenizer_config.json:   0%|          | 0.00/19.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]


=== SEED 42 | Fold 1/5 ===


pytorch_model.bin:   0%|          | 0.00/412M [00:00<?, ?B/s]

  epoch 1: loss=4.355  val加權F1=0.5333
  epoch 2: loss=3.388  val加權F1=0.5452
  epoch 3: loss=2.645  val加權F1=0.5862
  epoch 4: loss=2.005  val加權F1=0.5788
  epoch 5: loss=1.507  val加權F1=0.5927
  epoch 6: loss=1.171  val加權F1=0.6211
  epoch 7: loss=0.928  val加權F1=0.6195
  epoch 8: loss=0.785  val加權F1=0.6215

=== SEED 42 | Fold 2/5 ===
  epoch 1: loss=4.321  val加權F1=0.5055
  epoch 2: loss=3.249  val加權F1=0.5336
  epoch 3: loss=2.556  val加權F1=0.5399
  epoch 4: loss=1.998  val加權F1=0.5650
  epoch 5: loss=1.549  val加權F1=0.5727
  epoch 6: loss=1.252  val加權F1=0.5782
  epoch 7: loss=1.003  val加權F1=0.5833
  epoch 8: loss=0.853  val加權F1=0.5814

=== SEED 42 | Fold 3/5 ===
  epoch 1: loss=4.355  val加權F1=0.4678
  epoch 2: loss=3.307  val加權F1=0.5861
  epoch 3: loss=2.495  val加權F1=0.5496
  epoch 4: loss=1.896  val加權F1=0.5914
  epoch 5: loss=1.456  val加權F1=0.5879
  epoch 6: loss=1.113  val加權F1=0.6049
  epoch 7: loss=0.898  val加權F1=0.6001
  epoch 8: loss=0.752  val加權F1=0.6032

=== SEED 42 | Fold 4/5 ===
  ep

## Step 5｜本地 CV 分數


In [ ]:
cv,detail=weighted_macro_f1(train_data, decode_with_logic(oof_prob))
print(f"集成 CV 加權 Macro-F1 = {cv:.4f}\n")
for f in EVAL_FIELDS:
    print(f"  {f:24s} Macro-F1={detail[f]:.4f}  (權重 {FIELD_WEIGHTS[f]})")
print("\n各欄位細部報告：")
oof_pred=decode_with_logic(oof_prob)
for f in EVAL_FIELDS:
    print(f"\n--- {f} ---")
    print(classification_report([g[f] for g in train_data],[p[f] for p in oof_pred],labels=EVAL_FIELDS[f],zero_division=0))

集成 CV 加權 Macro-F1 = 0.6113

  promise_status           Macro-F1=0.8139  (權重 0.2)
  verification_timeline    Macro-F1=0.5676  (權重 0.15)
  evidence_status          Macro-F1=0.6957  (權重 0.3)
  evidence_quality         Macro-F1=0.4421  (權重 0.35)

各欄位細部報告：

--- promise_status ---
              precision    recall  f1-score   support

         Yes       0.93      0.94      0.93      1627
          No       0.72      0.67      0.69       373

    accuracy                           0.89      2000
   macro avg       0.82      0.80      0.81      2000
weighted avg       0.89      0.89      0.89      2000


--- verification_timeline ---
                       precision    recall  f1-score   support

              already       0.60      0.47      0.53       718
       within_2_years       0.36      0.35      0.36        34
between_2_and_5_years       0.54      0.65      0.59       498
    more_than_5_years       0.60      0.74      0.66       377
                  N/A       0.72      0.67      0.

## Step 6｜Macro-F1 門檻優化 + 產生提交檔
純後處理：用 OOF 搜尋每類別權重以最大化 Macro-F1，再套到測試集。
會同時輸出**未優化版**與**優化版**兩個提交檔，讓你比較後再決定上傳哪個。

⚠️ 注意：優化後 CV 有樂觀偏誤（同份資料調又評），真實線上增幅約為其一半。
⚠️ 排行榜顯示「最後一次」提交，上傳優化版會取代現有分數——請保留未優化版以備還原。


In [ ]:
TUNABLE = {
    "promise_status":        ["Yes","No"],
    "verification_timeline": ["already","within_2_years","between_2_and_5_years","more_than_5_years"],
    "evidence_status":       ["Yes","No"],
    "evidence_quality":      ["Clear","Not Clear"],   # 不調 Misleading(2筆)
}
GRID = [0.5,0.6,0.7,0.8,0.9,1.0,1.1,1.25,1.5,1.75,2.0]

def decode_weighted(probs, W):
    N=len(probs["promise_status"]); out=[]
    ps_ids=[label2id["promise_status"][l] for l in ["Yes","No"]]
    tl=["already","within_2_years","between_2_and_5_years","more_than_5_years"]; tl_ids=[label2id["verification_timeline"][l] for l in tl]
    es_lab=["Yes","No"]; es_ids=[label2id["evidence_status"][l] for l in es_lab]
    eq=["Clear","Not Clear","Misleading"]; eq_ids=[label2id["evidence_quality"][l] for l in eq]
    pP=probs["promise_status"]*W["promise_status"]; pT=probs["verification_timeline"]*W["verification_timeline"]
    pE=probs["evidence_status"]*W["evidence_status"]; pQ=probs["evidence_quality"]*W["evidence_quality"]
    for i in range(N):
        ps="Yes" if pP[i,ps_ids[0]]>=pP[i,ps_ids[1]] else "No"
        r={"promise_status":ps}
        if ps=="No":
            r.update(verification_timeline="N/A",evidence_status="N/A",evidence_quality="N/A")
        else:
            r["verification_timeline"]=tl[int(np.argmax(pT[i,tl_ids]))]
            e=es_lab[int(np.argmax(pE[i,es_ids]))]; r["evidence_status"]=e
            r["evidence_quality"]=eq[int(np.argmax(pQ[i,eq_ids]))] if e=="Yes" else "N/A"
        out.append(r)
    return out

def build_submission(pred, path):
    sub=pd.DataFrame([{"id":str(test_data[i]["id"]), **pred[i]} for i in range(len(test_data))])
    sub=sub[["id","promise_status","verification_timeline","evidence_status","evidence_quality"]]
    sub["id"]=sub["id"].astype(int); sub=sub.sort_values("id").reset_index(drop=True)
    assert len(sub)==2000
    assert list(sub["id"])==list(range(12001,14001))
    for f in EVAL_FIELDS: assert not (set(sub[f])-set(EVAL_FIELDS[f]))
    assert sub.isnull().sum().sum()==0
    m=sub["promise_status"]=="No"
    assert (sub.loc[m,["verification_timeline","evidence_status","evidence_quality"]]=="N/A").all().all()
    assert (sub.loc[sub["evidence_status"].isin(["No","N/A"]),"evidence_quality"]=="N/A").all()
    sub.to_csv(path, index=False, encoding="utf-8")
    return sub

# (1) 未優化版（保底）
build_submission(decode_with_logic(test_prob), f"{OUT_DIR}/submission_base.csv")

# (2) 門檻優化
W={f:np.ones(num_labels[f]) for f in EVAL_FIELDS}
base_cv,_=weighted_macro_f1(train_data, decode_weighted(oof_prob, W))
best=base_cv
for it in range(4):
    improved=False
    for f in EVAL_FIELDS:
        for lab in TUNABLE[f]:
            ci=label2id[f][lab]; bestv=W[f][ci]
            for v in GRID:
                W[f][ci]=v
                sc,_=weighted_macro_f1(train_data, decode_weighted(oof_prob, W))
                if sc>best+1e-6: best=sc; bestv=v; improved=True
            W[f][ci]=bestv
    if not improved: break
opt_cv,detail=weighted_macro_f1(train_data, decode_weighted(oof_prob, W))
sub_opt=build_submission(decode_weighted(test_prob, W), f"{OUT_DIR}/submission_opt.csv")

print(f"未優化 CV = {base_cv:.4f}")
print(f"優化後 CV = {opt_cv:.4f}  (+{opt_cv-base_cv:.4f})  ← 含樂觀偏誤，真實線上增幅約一半\n")
for f in EVAL_FIELDS:
    print(f"  {f:24s} {detail[f]:.4f}  | 權重 {np.round(W[f],2)}")
print("\n兩個檔都已輸出：submission_base.csv（保底）/ submission_opt.csv（優化）")
print("optimized promise 分佈:", sub_opt["promise_status"].value_counts().to_dict())

未優化 CV = 0.6113
優化後 CV = 0.6208  (+0.0095)  ← 含樂觀偏誤，真實線上增幅約一半

  promise_status           0.8177  | 權重 [1.25 1.1 ]
  verification_timeline    0.5819  | 權重 [1.1 1.5 1.1 1.1 1. ]
  evidence_status          0.7022  | 權重 [2. 1. 1.]
  evidence_quality         0.4553  | 權重 [1.25 1.   1.   1.  ]

兩個檔都已輸出：submission_base.csv（保底）/ submission_opt.csv（優化）
optimized promise 分佈: {'Yes': 1675, 'No': 325}
